<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_analysis/seq2one/stage_07_01a_mlp_seq2one_robustness_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_01a -  SEQ2ONE - Modelo MLP - Robustness Testing**

En esta sección iniciamos el proceso de **tuning del modelo MLP bajo el enfoque many-to-one (SEQ2ONE)**, utilizando como variable objetivo el **`delta_60`**, es decir, la variación en puntos del MNQ en los próximos 60 minutos.

El enfoque SEQ2ONE consiste en utilizar una ventana histórica de tamaño \( L \) minutos como entrada y predecir un único valor futuro correspondiente al horizonte seleccionado. En este caso:

$$
X \in \mathbb{R}^{(L \times F)} \quad \longrightarrow \quad y \in \mathbb{R}
$$

donde:

- $L$ = window size  
- $F$ = número de features (36)
- $y$ = `delta_60`  

---

**Motivación**

De acuerdo con el workflow de investigación y modelado presentado en el libro *Machine Learning for Algorithmic Trading*, el diseño y tuning del modelo corresponde a la etapa de:

> **Design, tune, and evaluate ML models to generate trading signals**

En esta fase, el objetivo no es todavía el backtesting completo, sino encontrar una configuración que:

- Generalice correctamente (early stopping sobre VALID)  
- Maximice capacidad predictiva (R², RMSE, MAE)  
- Mantenga estabilidad direccional (DA)  

---

**Objetivo del tuning**

El propósito del tuning será:

1. Evaluar distintos tamaños de ventana $L$.  
2. Ajustar hiperparámetros del MLP:
   - `hidden_dim`
   - `dropout`
   - `learning_rate`
   - `weight_decay`
3. Seleccionar la mejor configuración usando exclusivamente el conjunto **VALID**.  
4. Reservar el conjunto **TEST** para evaluación final no sesgada.  

Este procedimiento evita leakage y respeta el principio de generalización fuera de muestra, fundamental en modelado financiero.

---

**Contexto técnico**

El MLP:

- Recibe ventanas aplanadas en formato 2D.  
- No modela memoria temporal explícita (a diferencia de LSTM/GRU).  
- Aprende relaciones no lineales entre patrones recientes del mercado y el `delta_60`.  

Por lo tanto, el tamaño de ventana seleccionado determinará indirectamente cuánta información temporal puede capturar el modelo.

---

En las siguientes secciones se definirá el espacio de búsqueda y se ejecutará el proceso de tuning controlado sobre TRAIN/VALID.



# **BLOQUE DE EJECUCIÓN COMPLETO**

In [1]:
window_sizes = [90, 180]
targets = ['delta_60']
splits = ['train', 'valid', 'test']

## **1. Imports + paths**

In [2]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [4]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

In [5]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [6]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl')}

## **4. Reproducibilidad**

In [7]:
#def set_seeds(seed: int = 42) -> None:
#    random.seed(seed)
#    np.random.seed(seed)
#    os.environ["PYTHONHASHSEED"] = str(seed)
#
#set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [8]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [9]:
#print(compute_seq2one_metrics.__doc__)

## **6. Carga de data windows**

In [10]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y


In [11]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [12]:
from typing import Any, Dict, Mapping
from pathlib import Path

def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
    splits: tuple[str, ...] = ("train", "valid", "test"),
) -> Dict[str, Any]:
    """
    Carga X/y para los splits solicitados y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path (.npz con X,y)
    scalers_path[target] -> Path (scaler)
    """

    # --------------------------
    # 1) Validaciones base
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    valid_splits = {"train", "valid", "test"}
    splits_set = set(splits)
    unknown = splits_set - valid_splits
    if unknown:
        raise ValueError(f"splits inválidos: {sorted(unknown)}. Usar {sorted(valid_splits)}")

    # Validar que existan los splits solicitados para ese target
    available = set(windows_paths[window_size][target].keys())
    missing = splits_set - available
    if missing:
        raise KeyError(
            f"Faltan splits {sorted(missing)} en windows_paths[{window_size}]['{target}']. "
            f"Disponibles: {sorted(available)}"
        )

    # --------------------------
    # 2) Paths (solo los necesarios)
    # --------------------------
    split_paths: Dict[str, Path] = {sp: windows_paths[window_size][target][sp] for sp in splits}
    scaler_path = scalers_path[target]

    # --------------------------
    # 3) Carga por split
    # --------------------------
    out_splits: Dict[str, Dict[str, Any]] = {}
    out_paths: Dict[str, str] = {}

    for sp, p in split_paths.items():
        X, y = load_npz_windows(p)
        out_splits[sp] = {"X": X, "y": y}
        out_paths[sp] = str(p)

    scaler = load_scaler(scaler_path)
    out_paths["scaler"] = str(scaler_path)

    # --------------------------
    # 4) Inferir horizonte (robusto)
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception as e:
        raise ValueError(f"No se pudo inferir horizon desde target='{target}'. Esperado sufijo '_<int>'") from e

    # --------------------------
    # 5) Retorno
    # --------------------------
    out: Dict[str, Any] = {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": out_paths,
        "scaler": scaler,
    }
    out.update(out_splits)

    return out

In [13]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [14]:
def create_bundles(
    window_size,
    targets: list,
    windows_paths=windows_paths,
    scalers_paths=scalers_paths,
    *,
    flatten_X: bool = False,
    splits: tuple[str, ...] = ("train", "valid", "test"),
    verbose_shapes: bool = True,
    ):
    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
            splits=splits,
        )

        if flatten_X:
            for sp in splits:
                b[sp]["X"] = maybe_flatten_X(b[sp]["X"], flatten=True)

        bundles.append(b)

    if verbose_shapes:
        for b in bundles:
            for sp in splits:
                print(f"H{b['horizon']} {sp.capitalize():<5}:", b[sp]["X"].shape, b[sp]["y"].shape)
            print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [15]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [16]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [17]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

## **8. Métricas ML**

In [18]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

In [19]:
def get_metrics_torch(bundle, model, *, device) -> tuple[dict, dict]:
  # -------- VALID --------
  X_valid = bundle["valid"]["X"]
  y_valid = bundle["valid"]["y"]
  y_pred_valid = predict_mlp(model, X_valid, device=device)
  metrics_valid = compute_seq2one_metrics(y_valid, y_pred_valid, compute_r2=True)

  # -------- TEST --------
  X_test = bundle["test"]["X"]
  y_test = bundle["test"]["y"]
  y_pred_test = predict_mlp(model, X_test, device=device)
  metrics_test  = compute_seq2one_metrics(y_test, y_pred_test,  compute_r2=True)

  return metrics_valid, metrics_test

## **9. Gestión de dataset de métricas**

In [20]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [21]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

# **DEFINICIÓN DE MODELO**

## **11. Definición del modelo — placeholder**

### **11.1. Imports (PyTorch) + semillas**

In [22]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [23]:
import os
import random
import numpy as np
import torch

def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # determinismo (puede bajar performance)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # fuerza algoritmos deterministas (puede fallar si alguna op no tiene versión determinista)
    torch.use_deterministic_algorithms(True)

    # opcional, ayuda a determinismo en algunas matmuls CUDA (si usa GPU)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# NO lo llame aquí si luego hará loop de seeds; llámelo dentro del loop.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


### **11.2. Dataset/DataLoader desde bundle**

In [24]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def _seed_worker(worker_id: int) -> None:
    # asegura que numpy/random en cada worker quede determinista
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loaders_from_bundle(
    bundle: dict,
    *,
    splits: tuple[str, ...] = ("train", "valid", "test"),
    batch_size: int = 4096,
    num_workers: int = 0,
    pin_memory: bool | None = None,
    seed: int | None = None,
) -> dict:
    if pin_memory is None:
        pin_memory = torch.cuda.is_available()

    loaders: dict[str, DataLoader] = {}

    # generator para controlar el shuffle de forma reproducible
    g = None
    if seed is not None:
        g = torch.Generator()
        g.manual_seed(seed)

    for split in splits:
        if split not in bundle:
            continue

        X = np.asarray(bundle[split]["X"], dtype=np.float32)

        y = np.asarray(bundle[split]["y"], dtype=np.float32)
        if y.ndim == 1:
            y = y.reshape(-1, 1)

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
            drop_last=False,
            generator=g if shuffle else None,
            worker_init_fn=_seed_worker if num_workers > 0 else None,
            persistent_workers=(num_workers > 0),
        )

    return loaders

### **11.3. Definición del modelo MLP (simple y controlado)**

In [25]:
import torch
import torch.nn as nn

# DEFINICIÓN DEL MODELO MLP (robusta)
class MLPSeq2One(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 128, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

def build_mlp_from_bundle(
    bundle: dict,
    *,
    hidden_dim: int = 128,
    dropout: float = 0.0,
) -> MLPSeq2One:
    """
    Crea MLPSeq2One tomando in_dim desde bundle['train']['X'].
    Requiere que X esté 2D (flatten_X=True).
    """
    X = bundle["train"]["X"]
    if getattr(X, "ndim", None) != 2:
        raise ValueError(f"MLPSeq2One requiere X 2D (flatten_X=True). Recibido X.ndim={getattr(X,'ndim',None)}")
    in_dim = int(X.shape[1])
    return MLPSeq2One(in_dim=in_dim, hidden_dim=hidden_dim, dropout=dropout)

### **11.4. Entrenamiento con early stopping (VALID)**

In [26]:
import copy
import time
import torch
import torch.nn as nn
from typing import Optional
from torch.utils.data import DataLoader

def _ts():
    return time.strftime("%H:%M:%S")

@torch.no_grad()
def evaluate_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    mse_sum = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        pred = model(xb)
        mse_sum += torch.sum((pred - yb) ** 2).item()
        n += yb.numel()
    return mse_sum / max(n, 1)

def train_mlp(
    loaders: dict,
    *,
    in_dim: Optional[int] = None,
    hidden_dim: int = 128,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 1e-6,
    device: torch.device,
    verbose: bool = True,
    log_every: int = 0,
):
    if in_dim is None:
        xb0, _ = next(iter(loaders["train"]))
        in_dim = int(xb0.shape[1])

    if verbose:
        print(f"[{_ts()}] [TRAIN] START | in_dim={in_dim} hidden_dim={hidden_dim} dropout={dropout} "
              f"lr={lr} wd={weight_decay} max_epochs={max_epochs} patience={patience} min_delta={min_delta} "
              f"device={device.type}")

    t_global = time.perf_counter()

    model = MLPSeq2One(in_dim=in_dim, hidden_dim=hidden_dim, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_valid = float("inf")
    best_epoch = 0
    bad_epochs = 0

    epochs_ran = 0

    for epoch in range(1, max_epochs + 1):
        epochs_ran = epoch
        t_epoch = time.perf_counter()

        model.train()
        train_loss_sum = 0.0
        train_n = 0

        for b, (xb, yb) in enumerate(loaders["train"], start=1):
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            bs = yb.numel()
            train_loss_sum += loss.item() * bs
            train_n += bs

            if verbose and log_every and (b % log_every == 0):
                print(f"[{_ts()}]   epoch={epoch:02d} batch={b:04d} | train_loss_avg={(train_loss_sum/max(train_n,1)):.6f}")

        train_loss_avg = train_loss_sum / max(train_n, 1)

        t0 = time.perf_counter()
        valid_mse = evaluate_mse(model, loaders["valid"], device)
        dt_valid = time.perf_counter() - t0
        dt_epoch = time.perf_counter() - t_epoch

        improved = valid_mse < (best_valid - min_delta)
        if improved:
            best_valid = valid_mse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            flag = "BEST" if improved else f"no_improve({bad_epochs}/{patience})"
            print(f"[{_ts()}] epoch={epoch:02d} | train_loss={train_loss_avg:.6f} | "
                  f"valid_mse={valid_mse:.6f} | {flag} | dt_valid={dt_valid:.2f}s | dt_epoch={dt_epoch:.2f}s")

        if bad_epochs >= patience:
            if verbose:
                print(f"[{_ts()}] [TRAIN] EARLY STOPPING | patience={patience} | best_valid_mse={best_valid:.6f} @epoch={best_epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    info = {
        "best_valid_mse": float(best_valid),
        "best_epoch": int(best_epoch),
        "epochs_ran": int(epochs_ran),
        "final_lr": float(opt.param_groups[0]["lr"]),
    }

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [TRAIN] END | best_valid_mse={best_valid:.6f} @epoch={best_epoch} | dt_total={dt_all:.2f}s")

    return model, info

### **11.5. Predicciones MLP**


In [27]:
@torch.no_grad()
def predict_mlp(
    model,
    X: np.ndarray,
    *,
    device: torch.device,
    batch_size: int = 32768,
) -> np.ndarray:

    model.eval()

    X = np.asarray(X, dtype=np.float32)
    if X.ndim != 2:
        raise ValueError(f"MLPSeq2One requiere X 2D. Recibido X.ndim={X.ndim}")

    n = X.shape[0]
    preds = []

    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X[i:i+batch_size]).to(device, non_blocking=True)
        yb = model(xb)

        if yb.ndim == 2 and yb.shape[1] == 1:
            yb = yb[:, 0]

        preds.append(yb.cpu().numpy())

    return np.concatenate(preds, axis=0)

## **12. Ejecución completa**

In [28]:
# ============================================================
# run_transformer (con loop por seeds) + run_transformer_incremental (skip por seeds)
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Any, Dict, Tuple

import gc
import time
import pandas as pd
import torch


def run_mlp(
    window_size: int,
    *,
    n_features: int = 36,
    hidden_dim: int = 128,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    batch_size: int = 16384,
    max_epochs: int = 30,
    patience: int = 5,
    seeds: list[int] = (0, 1, 2, 3, 4),
    targets: list[str] = ("delta_60",),
    verbose: bool = True,
) -> pd.DataFrame:

    L = int(window_size)

    rows = []

    if verbose:
        print("\n" + "=" * 80)
        print(f"[{_ts()}] MLP | ROBUSTEZ SEEDS={list(seeds)} | WINDOW_SIZE=L{L} | in_dim={L*n_features} | "
              f"hd={hidden_dim} | dropout={dropout} | lr={lr} | w_decay={weight_decay}")
        print("=" * 80)

    t_global = time.perf_counter()

    for i, target in enumerate(targets, start=1):
        for j, seed in enumerate(seeds, start=1):
            t_run0 = time.perf_counter()

            # 0) SEED
            set_seed(seed)

            if verbose:
                print(f"[{_ts()}] [{i}/{len(targets)}] [{j}/{len(seeds)}] START target='{target}' | L{L} | seed={seed}")

            # 1) BUILD
            t0 = time.perf_counter()
            (bundle,) = create_bundles(
                window_size=L,
                targets=[target],
                windows_paths=windows_paths,
                scalers_paths=scalers_paths,
                flatten_X=True,
                splits=("train", "valid", "test"),
                verbose_shapes=False,
            )
            dt_build = time.perf_counter() - t0

            # 2) LOADERS (importante: pasar seed para shuffle reproducible)
            t0 = time.perf_counter()
            loaders = make_loaders_from_bundle(
                bundle,
                splits=("train", "valid", "test"),
                batch_size=batch_size,
                num_workers=0,
                seed=seed,  # <-- clave
            )
            dt_loaders = time.perf_counter() - t0

            # 3) TRAIN
            t0 = time.perf_counter()
            model, info = train_mlp(
                loaders,
                in_dim=L * n_features,
                hidden_dim=hidden_dim,
                dropout=dropout,
                lr=lr,
                weight_decay=weight_decay,
                max_epochs=max_epochs,
                patience=patience,
                device=device,
                verbose=verbose,
            )
            dt_train = time.perf_counter() - t0

            # 4) METRICS
            t0 = time.perf_counter()
            metrics_valid, metrics_test = get_metrics_torch(bundle, model, device=device)
            dt_eval = time.perf_counter() - t0

            # 5) DF (+ columnas de robustez)
            df_v = metrics_to_df(
                metrics_valid, model="mlp", split="valid",
                horizon=bundle["horizon"], window_size=bundle["window_size"], target=bundle["target"],
            )
            df_t = metrics_to_df(
                metrics_test, model="mlp", split="test",
                horizon=bundle["horizon"], window_size=bundle["window_size"], target=bundle["target"],
            )

            for df_ in (df_v, df_t):
                df_["seed"] = seed
                df_["best_valid_mse"] = info["best_valid_mse"]
                df_["best_epoch"] = info["best_epoch"]
                df_["epochs_ran"] = info["epochs_ran"]

            rows.append(df_v)
            rows.append(df_t)

            # cleanup
            del bundle, loaders, model, metrics_valid, metrics_test
            gc.collect()

            if verbose:
                dt_run = time.perf_counter() - t_run0
                print(f"[{_ts()}] DONE target='{target}' | L{L} | seed={seed} | "
                      f"dt_total={dt_run:.2f}s | build={dt_build:.2f}s | loaders={dt_loaders:.2f}s | "
                      f"train={dt_train:.2f}s | eval={dt_eval:.2f}s")

    df = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "seed", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [DONE] L{L} | rows={len(df)} | dt_total={dt_all:.2f}s")

    return df

In [29]:
from pathlib import Path
import time
import gc
import pandas as pd
import torch

def _ts() -> str:
    return time.strftime("%H:%M:%S")


def run_mlp_incremental(
    *,
    window_sizes: list[int],

    # ---- robustez ----
    seeds: int | list[int] = 42,

    # ---- hiperparámetros MLP ----
    n_features: int = 36,
    hidden_dim: int = 128,
    dropout: float = 0.0,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    batch_size: int = 16384,
    max_epochs: int = 30,
    patience: int = 5,

    # ---- targets ----
    targets: list[str] = ("delta_60",),

    # ---- persistencia ----
    name: str = "mlp",
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_testing",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Incremental para MLP (misma lógica que Transformer): guarda progreso POR SEED.

    Para cada L:
      - Detecta seeds completas (para cada target: splits {valid,test})
      - Ejecuta run_mlp(L, seeds=[seed], targets=targets) solo para seeds faltantes
      - Guarda df_hist inmediatamente tras cada seed
    """

    # --------- normalizar seeds ----------
    if isinstance(seeds, int):
        seeds_list = [int(seeds)]
    else:
        seeds_list = [int(s) for s in seeds]
    expected_seeds = set(seeds_list)

    expected_targets = list(targets)
    expected_splits = {"valid", "test"}

    metrics_dir = Path(base_dir)
    metrics_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = metrics_dir / f"seq2one_{name}_metrics.parquet"

    # --------- cargar histórico ----------
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    # --------- helper: seeds completas por L ----------
    def _complete_seeds_for_L(df: pd.DataFrame, L: int) -> set[int]:
        if df.empty:
            return set()

        required = {"seed", "split", "target", "window_size", "model"}
        if not required.issubset(df.columns):
            return set()

        dfL = df[
            (df["model"] == name) &
            (df["window_size"] == int(L)) &
            (df["target"].isin(expected_targets)) &
            (df["seed"].isin(expected_seeds)) &
            (df["split"].isin(expected_splits))
        ].copy()

        if dfL.empty:
            return set()

        # Para cada (seed, target): splits presentes
        splits_by_seed_target = (
            dfL.groupby(["seed", "target"])["split"]
               .apply(lambda s: set(s.tolist()))
               .to_dict()
        )

        complete = set()
        for s in expected_seeds:
            ok = True
            for t in expected_targets:
                sp = splits_by_seed_target.get((s, t), set())
                if not expected_splits.issubset(sp):
                    ok = False
                    break
            if ok:
                complete.add(int(s))
        return complete

    # --------- loop por window_size ----------
    for L in window_sizes:
        L = int(L)

        complete_seeds = _complete_seeds_for_L(df_hist, L)
        missing_seeds = set(expected_seeds) - complete_seeds

        if not missing_seeds:
            if verbose:
                print(f"[SKIP] {name} L={L} targets={expected_targets} ya existe completo para seeds={sorted(expected_seeds)}")
            continue

        if verbose:
            print(f"[RUN] {name} L={L} targets={expected_targets} faltan seeds={sorted(missing_seeds)}")

        # --------- ejecutar y GUARDAR por seed ----------
        for seed in sorted(missing_seeds):
            if verbose:
                print(f"[RUN] {name} L={L} -> seed={seed} (guardado inmediato)")

            df_seed = None
            try:
                df_seed = run_mlp(
                    window_size=L,
                    n_features=n_features,
                    hidden_dim=hidden_dim,
                    dropout=dropout,
                    lr=lr,
                    weight_decay=weight_decay,
                    batch_size=batch_size,
                    max_epochs=max_epochs,
                    patience=patience,
                    seeds=[seed],                 # <- 1 seed por corrida
                    targets=list(expected_targets),
                    verbose=verbose,
                )

                # asegurar nombre del modelo en el DF (por si run_mlp usa "mlp" fijo)
                df_seed["model"] = name

                # anexar a histórico
                if df_hist.empty:
                    df_hist = df_seed.copy()
                else:
                    df_hist = pd.concat([df_hist, df_seed], ignore_index=True)

                # guardar inmediatamente (checkpoint)
                save_seq2one_metrics(df_hist, name=name, base_dir=base_dir)

            finally:
                df_seed = None
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            # recargar desde disco (opcional pero robusto)
            if metrics_path.exists():
                df_hist = pd.read_parquet(metrics_path)

    return (
        df_hist.sort_values(["window_size", "target", "seed", "split", "horizon_min", "model"])
              .reset_index(drop=True)
    )

In [30]:
seeds = [
    1, 7, 42, 123, 999,
    2024, 31415, 27182, 8080, 777,
    5555, 8888, 1001, 2025, 9090,
    3333, 4444, 6666, 1212, 2121
]

In [31]:
df_mlp_all_sizes = run_mlp_incremental(
    window_sizes=[90, 180],
    hidden_dim=128,
    dropout=0.0,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=16384,
    max_epochs=30,
    patience=5,

    # robustez
    seeds =seeds,

    targets=["delta_60"],
    name="mlp",
    verbose=True,
)

[RUN] mlp L=90 targets=['delta_60'] faltan seeds=[777, 1001, 1212, 2024, 2025, 2121, 3333, 4444, 5555, 6666, 8080, 8888, 9090, 27182, 31415]
[RUN] mlp L=90 -> seed=777 (guardado inmediato)

[16:54:44] MLP | ROBUSTEZ SEEDS=[777] | WINDOW_SIZE=L90 | in_dim=3240 | hd=128 | dropout=0.0 | lr=0.001 | w_decay=0.0001
[16:54:49] [1/1] [1/1] START target='delta_60' | L90 | seed=777
[16:55:16] [TRAIN] START | in_dim=3240 hidden_dim=128 dropout=0.0 lr=0.001 wd=0.0001 max_epochs=30 patience=5 min_delta=1e-06 device=cuda
[16:55:28] epoch=01 | train_loss=2317.992253 | valid_mse=1913.367234 | BEST | dt_valid=1.80s | dt_epoch=11.09s
[16:55:38] epoch=02 | train_loss=2020.343209 | valid_mse=1794.138605 | BEST | dt_valid=1.67s | dt_epoch=9.78s
[16:55:47] epoch=03 | train_loss=1915.813523 | valid_mse=1712.008028 | BEST | dt_valid=1.65s | dt_epoch=9.77s
[16:55:57] epoch=04 | train_loss=1834.711867 | valid_mse=1660.366093 | BEST | dt_valid=1.67s | dt_epoch=9.79s
[16:56:07] epoch=05 | train_loss=1774.760147 |

In [32]:
df_mlp_all_sizes

,model,split,window_size,target,horizon_min,seed,MAE,RMSE,R2,DA,best_valid_mse,best_epoch,epochs_ran,hidden_dim,dropout,lr,w_decay,batch_size,max_epochs,patience
0,mlp,test,90,delta_60,60,1,37.712649,63.676674,0.396801,0.724112,1332.914070,30,30,128.0,0.0,0.001,0.0001,16384.0,30.0,5.0
1,mlp,valid,90,delta_60,60,1,22.561485,36.509095,0.438766,0.740451,1332.914070,30,30,128.0,0.0,0.001,0.0001,16384.0,30.0,5.0
2,mlp,test,90,delta_60,60,7,37.168299,63.174333,0.406280,0.730049,1324.642699,28,30,128.0,0.0,0.001,0.0001,16384.0,30.0,5.0
3,mlp,valid,90,delta_60,60,7,22.215046,36.395641,0.442248,0.747520,1324.642699,28,30,128.0,0.0,0.001,0.0001,16384.0,30.0,5.0
4,mlp,test,90,delta_60,60,42,37.816548,63.500839,0.400127,0.717732,1341.256033,30,30,128.0,0.0,0.001,0.0001,16384.0,30.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,mlp,valid,180,delta_60,60,9090,21.794784,35.359527,0.449691,0.746078,1250.296107,30,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,mlp,test,180,delta_60,60,27182,36.670976,62.931564,0.419896,0.728543,1256.421570,29,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,mlp,valid,180,delta_60,60,27182,21.926187,35.446037,0.446995,0.740806,1256.421570,29,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,mlp,test,180,delta_60,60,31415,37.432384,62.141304,0.434374,0.729410,1331.760936,11,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## **13. Análisis**

In [33]:
df = df_mlp_all_sizes.copy()

### **13.1. Análisis descriptivo comparativo por tamaño de ventana (L=90 vs L=180)**

El primer análisis (mínimo y más informativo) es un resumen estadístico por window_size y split para comparar promedio y dispersión (robustez) de las métricas.

In [34]:
metrics = ["MAE", "RMSE", "R2", "DA"]

summary = (
    df.groupby(["window_size", "split"])[metrics]
      .agg(["mean", "std", "min", "max", "count"])
      .round(6)
)

summary

MAE                                             RMSE  \
                        mean       std        min        max count       mean   
window_size split                                                               
90          test   37.646452  0.550208  36.882175  38.898928    20  63.561111   
            valid  22.515627  0.243985  22.008496  23.030663    20  36.545031   
180         test   37.004239  0.485296  36.333443  38.301526    20  62.044767   
            valid  21.994106  0.417231  21.508260  23.085520    20  35.639616   

                                                               R2            \
                        std        min        max count      mean       std   
window_size split                                                             
90          test   0.507210  62.737400  64.809396    20  0.398952  0.009618   
            valid  0.175253  36.190014  36.978617    20  0.437648  0.005400   
180         test   0.919954  60.656982  63.948908    20  0.436012  0.016759   
            valid  0.438010  35.181841  36.614402    20  0.440858  0.013830   

                                                   DA                      \
                        min       max count      mean       std       min   
window_size split                                                           
90          test   0.375150  0.414465    20  0.724932  0.008233  0.705415   
            valid  0.424237  0.448533    20  0.743051  0.004170  0.732934   
180         test   0.400989  0.461072    20  0.725987  0.007577  0.708304   
            valid  0.409938  0.455208    20  0.742657  0.004548  0.733305   

                                   
                        max count  
window_size split                  
90          test   0.737180    20  
            valid  0.750209    20  
180         test   0.735673    20  
            valid  0.748507    20

**Análisis descriptivo comparativo por tamaño de ventana (20 seeds)**

**1. Desempeño promedio en TEST**

- MAE  
  - L=90 → 37.65  
  - L=180 → 37.00  
  - Mejora de aproximadamente 0.64 puntos.

- RMSE  
  - L=90 → 63.56  
  - L=180 → 62.04  
  - Mejora consistente (~1.5 puntos).

- R²  
  - L=90 → 0.3989  
  - L=180 → 0.4360  
  - Incremento de +0.037.

- Directional Accuracy (DA)  
  - L=90 → 0.7249  
  - L=180 → 0.7260  
  - Prácticamente iguales.

- Conclusión TEST:  
  - La ventana L=180 domina a L=90 en MAE, RMSE y R².  
  - La mejora se da principalmente en magnitud del error y varianza explicada.  
  - La mejora en DA es marginal.

**2. Robustez en TEST (desviación estándar)**

- R² (std)  
  - L=90 → 0.0096  
  - L=180 → 0.0168  

- MAE (std)  
  - L=90 → 0.55  
  - L=180 → 0.49  

La dispersión es baja en ambos casos.  
L=180 presenta mayor variabilidad en R², pero sigue siendo estable en términos absolutos.  
No hay evidencia de inestabilidad estructural al aumentar la ventana.

**3. Desempeño promedio en VALID**

- MAE  
  - L=90 → 22.52  
  - L=180 → 21.99  

- RMSE  
  - L=90 → 36.55  
  - L=180 → 35.64  

- R²  
  - L=90 → 0.4376  
  - L=180 → 0.4409  

- DA  
  - L=90 → 0.7431  
  - L=180 → 0.7427  

- Conclusión VALID:  
  - La ventana L=180 también mejora en validación, especialmente en MAE y RMSE.  
  - La mejora en R² es leve pero consistente.  
  - No se observa incremento de sobreajuste.

**4. Gap VALID–TEST (generalización)**

- L=90  
  - VALID R² = 0.438  
  - TEST R² = 0.399  
  - Gap ≈ 0.039  

- L=180  
  - VALID R² = 0.441  
  - TEST R² = 0.436  
  - Gap ≈ 0.005  

- El gap se reduce significativamente al usar ventana más larga.  
- L=180 presenta una alineación mucho más consistente entre validación y test.

**5. Conclusión general**

Con 20 seeds:

- L=180 mejora consistentemente el desempeño promedio en TEST.
- No aumenta de forma relevante la varianza entre inicializaciones.
- Reduce el gap VALID–TEST.
- No incrementa el overfitting.
- Aporta señal útil adicional respecto a L=90.

En MLP, al igual que en Transformer, la ventana L=180 domina estructuralmente a L=90.

### **13.2. Test estadístico formal**

Queremos responder:

> ¿La mejora en R² en TEST al pasar de L=90 a L=180 es estadísticamente significativa?

Como usaste las mismas seeds, el test correcto es:
- Paired t-test (muestras dependientes)
- Alternativamente Wilcoxon (no paramétrico)

**1. Preparar los vectores de R² (TEST)**

In [35]:
import numpy as np

# Filtrar solo TEST
df_test = df[df["split"] == "test"]

r2_90 = (
    df_test[df_test["window_size"] == 90]
    .sort_values("seed")["R2"]
    .values
)

r2_180 = (
    df_test[df_test["window_size"] == 180]
    .sort_values("seed")["R2"]
    .values
)

r2_90, r2_180

(array([0.39680066, 0.40628031, 0.40012737, 0.39066445, 0.39993731,
        0.40231521, 0.39341538, 0.40113697, 0.4090544 , 0.39304419,
        0.39354109, 0.41446461, 0.40275449, 0.40238349, 0.40648354,
        0.37514958, 0.41128013, 0.40568332, 0.39247381, 0.38204438]),
 array([0.42569593, 0.40639195, 0.44024954, 0.42458458, 0.43003913,
        0.46028671, 0.44843491, 0.45166353, 0.43163323, 0.44114893,
        0.45428951, 0.40098854, 0.44085636, 0.46107236, 0.43878954,
        0.43339775, 0.45605029, 0.42039733, 0.41989592, 0.43437371]))

**2. Paired t-test**

In [36]:
from scipy.stats import ttest_rel

t_stat, p_value = ttest_rel(r2_180, r2_90)
print("Paired t-test:")
t_stat, p_value

Paired t-test:


(np.float64(8.226172651433805), np.float64(1.1049837454735205e-07))

**3. Wilcoxon (más robusto con pocas muestras)**

In [37]:
from scipy.stats import wilcoxon

w_stat, p_wilcoxon = wilcoxon(r2_180, r2_90)
print("Wilcoxon:")
w_stat, p_wilcoxon

Wilcoxon:


(np.float64(2.0), np.float64(5.7220458984375e-06))

**Análisis estadístico formal: L=90 vs L=180 (R² en TEST, 20 seeds)**


**1. Paired t-test**

- Estadístico t = 8.2262  
- p-value = 1.10e-07  

Interpretación:  
El p-value es extremadamente menor que 0.05.  
Se rechaza con alta contundencia la hipótesis nula de igualdad de medias.

Existe una diferencia estadísticamente significativa entre L=90 y L=180.

**2. Test no paramétrico de Wilcoxon**

- Estadístico W = 2.0  
- p-value = 5.72e-06  

Interpretación:  
También muy por debajo de 0.05.  
La diferencia es significativa incluso sin asumir normalidad.

**3. Conclusión estadística**

Ambos tests confirman que:

- La mejora de R² al pasar de L=90 a L=180  
- No es producto del azar de inicialización  
- Es altamente consistente a través de las 20 seeds  

La magnitud del estadístico t (8.23) indica que la diferencia es no solo significativa, sino fuerte en términos estadísticos.

Por lo tanto, con el mismo modelo e hiperparámetros, la ventana de 180 muestras es estadísticamente superior a 90
para el target delta_60 en el caso del MLP.


### **13.3. Evaluación del tamaño del efecto (Effect Size – Cohen’s d)**

Hasta ahora demostramos que la diferencia entre L=90 y L=180 es estadísticamente significativa (p < 0.05).

Sin embargo, la significancia estadística solo responde a la pregunta:

- ¿La diferencia existe?

No responde a la pregunta más importante desde el punto de vista práctico:

- ¿La diferencia es grande o relevante?

El tamaño del efecto (Cohen’s d) mide la magnitud real de la diferencia entre ambos modelos en relación con la variabilidad entre seeds.

En términos simples:

- Si d es pequeño → la diferencia existe, pero su impacto es débil.
- Si d es moderado → la mejora es relevante.
- Si d es grande → el cambio de ventana tiene un impacto fuerte y consistente.

Objetivo en este análisis:

Cuantificar qué tan importante es la mejora en R² al pasar de ventana 90 a ventana 180, más allá de que sea estadísticamente significativa.

In [40]:
import numpy as np

# 1) Tomar R² en TEST y alinear por seed
df_test = df[df["split"] == "test"].copy()

r2_90 = (
    df_test[df_test["window_size"] == 90]
    .sort_values("seed")["R2"]
    .to_numpy()
)

r2_180 = (
    df_test[df_test["window_size"] == 180]
    .sort_values("seed")["R2"]
    .to_numpy()
)

# 2) Diferencias pareadas (L180 - L90)
diff = r2_180 - r2_90

# 3) Cohen's d para muestras pareadas (dz): media(diff) / std(diff)
d_z = diff.mean() / diff.std(ddof=1)

# 4) Resumen útil
out = {
    "n": int(diff.size),
    "mean_R2_90": float(r2_90.mean()),
    "mean_R2_180": float(r2_180.mean()),
    "mean_diff": float(diff.mean()),
    "std_diff": float(diff.std(ddof=1)),
    "cohens_dz": float(d_z),
}

out

{'n': 20,
 'mean_R2_90': 0.39895173459859623,
 'mean_R2_180': 0.4360119871080457,
 'mean_diff': 0.03706025250944944,
 'std_diff': 0.020147703527713386,
 'cohens_dz': 1.839428124325567}

**Tamaño del efecto (Cohen’s d – muestras pareadas)**


- Resultados:

  - n = 20 seeds  
  - R² medio L=90 = 0.39895  
  - R² medio L=180 = 0.43601  
  - Diferencia media = +0.03706  
  - Desvío estándar de las diferencias = 0.02015  
  - Cohen’s d (dz) = 1.84  

- Interpretación del tamaño del efecto:

  Reglas generales para Cohen’s d:
  - 0.2 → efecto pequeño  
  - 0.5 → efecto moderado  
  - 0.8 → efecto grande  

  En este caso: `d = 1.84`  
  Esto indica un efecto extremadamente grande.

- Conclusión práctica:

  - La mejora al pasar de L=90 a L=180 no solo es estadísticamente significativa.  
  - La magnitud del efecto es muy grande.  
  - La diferencia no es marginal ni dependiente de algunas seeds aisladas.  
  - La mejora es consistente y estructural en prácticamente todas las inicializaciones.

- Interpretación global (significancia + efecto):

  - Existe diferencia real (p << 0.05).  
  - La diferencia no es trivial.  
  - El MLP está explotando el contexto adicional de 180 muestras de manera muy eficiente.  
  - El incremento de memoria temporal aporta señal relevante y estable.

En el caso del MLP, el efecto de aumentar la ventana es incluso más fuerte que en Transformer en términos relativos.


### **13.4. Análisis de la distribución de las diferencias por seed**

Hasta ahora sabemos que:

- L=180 es mejor en promedio.
- La diferencia es estadísticamente significativa.
- El tamaño del efecto es muy grande (d = 1.84).

Pero aún falta responder una pregunta clave:

> ¿La mejora ocurre de manera consistente en casi todas las seeds, o está siendo impulsada por unas pocas inicializaciones extremadamente favorables?

En este punto buscamos:

- Analizar la distribución de las diferencias individuales (R²_180 − R²_90).
- Ver cuántas seeds realmente mejoran.
- Evaluar si la mejora es homogénea o depende de casos extremos.

**Objetivo concreto:**

Confirmar que la superioridad de L=180 es estructural y no producto de unas pocas semillas atípicas.

In [39]:
import pandas as pd
import numpy as np

# --- Recalcular por claridad ---
df_test = df[df["split"] == "test"].copy()

r2_90 = (
    df_test[df_test["window_size"] == 90]
    .set_index("seed")["R2"]
)

r2_180 = (
    df_test[df_test["window_size"] == 180]
    .set_index("seed")["R2"]
)

df_diff = pd.DataFrame({
    "R2_90": r2_90,
    "R2_180": r2_180,
})

df_diff["diff"] = df_diff["R2_180"] - df_diff["R2_90"]

# ============================================================
# 1) Tabla ordenada
# ============================================================

df_sorted = (
    df_diff
    .sort_values("diff", ascending=False)
    .round(6)
)

print("\n=== DIFERENCIAS POR SEED (ordenado por mejora) ===")
display(df_sorted)

# ============================================================
# 2) Resumen estadístico
# ============================================================

summary = (
    df_diff["diff"]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .to_frame()
    .T
    .round(6)
)

print("\n=== RESUMEN DE DIFERENCIAS (L180 - L90) ===")
display(summary)

# ============================================================
# 3) Conteo de mejoras
# ============================================================

n = len(df_diff)
n_pos = (df_diff["diff"] > 0).sum()
n_neg = (df_diff["diff"] < 0).sum()

improvement = pd.DataFrame([{
    "n_total": n,
    "n_mejora": n_pos,
    "n_empeora": n_neg,
    "pct_mejora": round(n_pos / n, 4),
    "pct_empeora": round(n_neg / n, 4),
}])

print("\n=== CONSISTENCIA DE LA MEJORA ===")
display(improvement)


=== DIFERENCIAS POR SEED (ordenado por mejora) ===


,R2_90,R2_180,diff
seed,,,
2121,0.393541,0.454290,0.060748
5555,0.402383,0.461072,0.058689
8080,0.375150,0.433398,0.058248
999,0.402315,0.460287,0.057971
1001,0.393415,0.448435,0.055020
31415,0.382044,0.434374,0.052329
1212,0.401137,0.451664,0.050527
2025,0.393044,0.441149,0.048105
8888,0.411280,0.456050,0.044770



=== RESUMEN DE DIFERENCIAS (L180 - L90) ===


,count,mean,std,min,median,max
diff,20.0,0.03706,0.020148,-0.013476,0.039112,0.060748



=== CONSISTENCIA DE LA MEJORA ===


,n_total,n_mejora,n_empeora,pct_mejora,pct_empeora
0,20,19,1,0.95,0.05


**Análisis de la distribución de las diferencias por seed (MLP)**

**1. Consistencia de la mejora**

- Total de seeds: 20  
- Seeds donde L=180 mejora a L=90: 19  
- Seeds donde L=180 empeora respecto a L=90: 1  
- Proporción de mejora: 95%  
- Proporción de deterioro: 5%  

Interpretación:  
- La mejora es extraordinariamente consistente.  
- En 19 de 20 inicializaciones, la ventana 180 supera a la 90.  
- No estamos frente a una mejora marginal ni dependiente de pocos casos favorables.


**2. Magnitud de las diferencias**

- Diferencia media: +0.0371  
- Mediana: +0.0391  
- Desvío estándar: 0.0201  
- Mejor mejora observada: +0.0607  
- Peor deterioro observado: −0.0135  

Observaciones clave:

- La mediana es positiva y muy cercana a la media → la mejora es homogénea.
- El desvío estándar es bajo → no hay comportamiento errático entre seeds.
- El peor caso negativo es pequeño (−0.0135).
- Las mejoras máximas (~+0.06) son varias y consistentes, no un único outlier.


**3. Estructura del único caso negativo**

Existe 1 seed donde L=180 empeora levemente respecto a L=90:

- Seed 3333: −0.0135

Sin embargo:

- El deterioro es pequeño en magnitud.
- No altera la media de forma relevante.
- No genera asimetría estructural en la distribución.

Esto refuerza que el efecto no depende de unos pocos casos extremos.

**4. Conclusión estructural**

La superioridad de L=180 en el MLP:

- No es producto de una única seed favorable.
- No depende de casos extremos aislados.
- Es consistente en prácticamente todas las inicializaciones (95%).
- Presenta distribución homogénea de mejoras.
- Tiene deterioros mínimos y poco relevantes.

La mejora es estructural, robusta y estable frente a la variabilidad de inicialización.

En el caso del MLP, la evidencia de consistencia es incluso más fuerte que en el Transformer.


### **13.5. Conclusión final del análisis comparativo L=90 vs L=180 (MLP – delta_60)**

El análisis realizado es metodológicamente completo y robusto:

- Se evaluaron 20 seeds independientes.
- Se realizó comparación descriptiva por ventana y split.
- Se aplicaron tests estadísticos formales (t-test pareado y Wilcoxon).
- Se calculó tamaño del efecto (Cohen’s d).
- Se analizó la distribución de diferencias por seed.
- Se evaluó la consistencia de la mejora (95% de las seeds mejoran).
- Se revisó el gap de generalización (valid vs test).
- La conclusión se basa en evidencia estadística y estructural consistente, no en observaciones puntuales.

Conclusión técnica:

La ventana L=180 es superior a L=90 para el target delta_60 en el MLP.

- La mejora es altamente significativa (p << 0.05).
- El tamaño del efecto es muy grande (d ≈ 1.84).
- La mejora es consistente en prácticamente todas las inicializaciones (19 de 20).
- El gap VALID–TEST se reduce notablemente.
- No se observa incremento del overfitting.
- El deterioro en el peor caso es pequeño y no altera la conclusión global.
- La mejora no depende de seeds extremas ni de outliers.

En consecuencia, el análisis puede considerarse formalmente cerrado y la ventana L=180 puede adoptarse como configuración preferente para el MLP en este target.
